In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.covariance import LedoitWolf
from itertools import product


import config.settings as settings
import src.produccion_x_equipo as crea_produccion
import src.extract_files as extract
import src.filter as filtra
import src.fc_prophet as fcProphet
#import src.produccion_x_equipo as produccion_x_equipo
import importlib
importlib.reload(filtra)
importlib.reload(fcProphet)
#import src.notebooks.TestFunctions as testfunctions
%load_ext autoreload
%autoreload 2


x is: 1


/home/jorge/Projects/Radioterapia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
input_path = settings.DATA_DIR / "processed" / "produccion_radioterapia.csv"

df = pd.read_csv(input_path)

In [3]:
print(df.head(5))
print(df.groupby("tipo_dia")["ds"].size())

           ds    equipo   y  capacidad tipo_dia  novedad
0  2022-05-03   HALCYON  59         70  laboral  ninguna
1  2022-05-03   SYNERGY  41         65  laboral  ninguna
2  2022-05-03  SYNERGY2  32         47  laboral  ninguna
3  2022-05-03  SYNERGY3  34         47  laboral  ninguna
4  2022-05-04   HALCYON  64         70  laboral  ninguna
tipo_dia
festivo           347
fin de semana    2670
laboral          6347
Name: ds, dtype: int64


In [4]:
dt = df.copy()
dt ["ds"] = pd.to_datetime(dt["ds"])
print(dt.head(5))
print(dt.groupby("tipo_dia")["ds"].size())

          ds    equipo   y  capacidad tipo_dia  novedad
0 2022-05-03   HALCYON  59         70  laboral  ninguna
1 2022-05-03   SYNERGY  41         65  laboral  ninguna
2 2022-05-03  SYNERGY2  32         47  laboral  ninguna
3 2022-05-03  SYNERGY3  34         47  laboral  ninguna
4 2022-05-04   HALCYON  64         70  laboral  ninguna
tipo_dia
festivo           347
fin de semana    2670
laboral          6347
Name: ds, dtype: int64


In [5]:
dt = pd.get_dummies(dt, columns = ["tipo_dia","novedad"])

In [6]:
dt.columns = (
    dt.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+","_",regex=True)
    .str.replace(r"tipo_dia_","",regex=True)
    .str.replace("novedad_","",regex=False)
)
print(dt)

             ds      equipo   y  capacidad  festivo  fin_de_semana  laboral  \
0    2022-05-03     HALCYON  59         70    False          False     True   
1    2022-05-03     SYNERGY  41         65    False          False     True   
2    2022-05-03    SYNERGY2  32         47    False          False     True   
3    2022-05-03    SYNERGY3  34         47    False          False     True   
4    2022-05-04     HALCYON  64         70    False          False     True   
...         ...         ...  ..        ...      ...            ...      ...   
9359 2026-04-30  EDGE2_6137   0          0     True          False    False   
9360 2026-04-30     HALCYON   0          0     True          False    False   
9361 2026-04-30    HALCYON3   0          0     True          False    False   
9362 2026-04-30    HALCYON4   0          0     True          False    False   
9363 2026-04-30    HALCYON5   0          0     True          False    False   

      control_de_calidad  falla  mantenimiento  nin

In [7]:
range_dates = pd.date_range(dt["ds"].min(), dt["ds"].max(), freq = "D")
equipos = sorted(dt["equipo"].unique())

complete_index = pd.MultiIndex.from_product(
    [range_dates, equipos],
    names = ["ds", "equipo"]
)

dt = (dt.set_index(["ds", "equipo"])
      .reindex(complete_index)
      .reset_index()
     )

In [8]:
print(dt.columns)
print(dt.head(5))

Index(['ds', 'equipo', 'y', 'capacidad', 'festivo', 'fin_de_semana', 'laboral',
       'control_de_calidad', 'falla', 'mantenimiento', 'ninguna'],
      dtype='object')
          ds      equipo     y  capacidad festivo fin_de_semana laboral  \
0 2022-05-03  EDGE1_6107   NaN        NaN     NaN           NaN     NaN   
1 2022-05-03  EDGE2_6137   NaN        NaN     NaN           NaN     NaN   
2 2022-05-03     HALCYON  59.0       70.0   False         False    True   
3 2022-05-03    HALCYON3   NaN        NaN     NaN           NaN     NaN   
4 2022-05-03    HALCYON4   NaN        NaN     NaN           NaN     NaN   

  control_de_calidad  falla mantenimiento ninguna  
0                NaN    NaN           NaN     NaN  
1                NaN    NaN           NaN     NaN  
2              False  False         False    True  
3                NaN    NaN           NaN     NaN  
4                NaN    NaN           NaN     NaN  


In [9]:
dt = dt.drop(columns = ["laboral","ninguna"])
print(dt.head(5))

          ds      equipo     y  capacidad festivo fin_de_semana  \
0 2022-05-03  EDGE1_6107   NaN        NaN     NaN           NaN   
1 2022-05-03  EDGE2_6137   NaN        NaN     NaN           NaN   
2 2022-05-03     HALCYON  59.0       70.0   False         False   
3 2022-05-03    HALCYON3   NaN        NaN     NaN           NaN   
4 2022-05-03    HALCYON4   NaN        NaN     NaN           NaN   

  control_de_calidad  falla mantenimiento  
0                NaN    NaN           NaN  
1                NaN    NaN           NaN  
2              False  False         False  
3                NaN    NaN           NaN  
4                NaN    NaN           NaN  


In [10]:
dt.groupby('equipo')['ds'].size()

equipo
EDGE1_6107    1459
EDGE2_6137    1459
HALCYON       1459
HALCYON3      1459
HALCYON4      1459
HALCYON5      1459
SYNERGY       1459
SYNERGY2      1459
SYNERGY3      1459
Name: ds, dtype: int64

In [11]:
dt["ds"] = pd.to_datetime(dt["ds"])
print(dt.columns)


Index(['ds', 'equipo', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


In [12]:
print(dt.head(5))

          ds      equipo     y  capacidad festivo fin_de_semana  \
0 2022-05-03  EDGE1_6107   NaN        NaN     NaN           NaN   
1 2022-05-03  EDGE2_6137   NaN        NaN     NaN           NaN   
2 2022-05-03     HALCYON  59.0       70.0   False         False   
3 2022-05-03    HALCYON3   NaN        NaN     NaN           NaN   
4 2022-05-03    HALCYON4   NaN        NaN     NaN           NaN   

  control_de_calidad  falla mantenimiento  
0                NaN    NaN           NaN  
1                NaN    NaN           NaN  
2              False  False         False  
3                NaN    NaN           NaN  
4                NaN    NaN           NaN  


In [13]:
dt = dt.sort_values(["equipo", "ds"])
print(dt.head(5))

           ds      equipo   y  capacidad festivo fin_de_semana  \
0  2022-05-03  EDGE1_6107 NaN        NaN     NaN           NaN   
9  2022-05-04  EDGE1_6107 NaN        NaN     NaN           NaN   
18 2022-05-05  EDGE1_6107 NaN        NaN     NaN           NaN   
27 2022-05-06  EDGE1_6107 NaN        NaN     NaN           NaN   
36 2022-05-07  EDGE1_6107 NaN        NaN     NaN           NaN   

   control_de_calidad falla mantenimiento  
0                 NaN   NaN           NaN  
9                 NaN   NaN           NaN  
18                NaN   NaN           NaN  
27                NaN   NaN           NaN  
36                NaN   NaN           NaN  


In [14]:
dt["capacidad"] = (
    dt.groupby("equipo")["capacidad"]
    .ffill()
    .bfill()
    .fillna(0)
    .astype(int)
)

dt["y"] = (
    dt.groupby("equipo")["y"]
    .ffill()
    .bfill()
    .fillna(0)
    .astype(int)
)




In [15]:
for col in ["festivo", "fin_de_semana", "control_de_calidad", "falla", "mantenimiento"]:
    if col in dt.columns:
        dt[col] = dt[col].astype("boolean").fillna(False).astype(bool)

In [16]:
print(dt.head(5))

           ds      equipo  y  capacidad  festivo  fin_de_semana  \
0  2022-05-03  EDGE1_6107  2         21    False          False   
9  2022-05-04  EDGE1_6107  2         21    False          False   
18 2022-05-05  EDGE1_6107  2         21    False          False   
27 2022-05-06  EDGE1_6107  2         21    False          False   
36 2022-05-07  EDGE1_6107  2         21    False          False   

    control_de_calidad  falla  mantenimiento  
0                False  False          False  
9                False  False          False  
18               False  False          False  
27               False  False          False  
36               False  False          False  


In [17]:
dt.groupby('equipo')['ds'].size()

equipo
EDGE1_6107    1459
EDGE2_6137    1459
HALCYON       1459
HALCYON3      1459
HALCYON4      1459
HALCYON5      1459
SYNERGY       1459
SYNERGY2      1459
SYNERGY3      1459
Name: ds, dtype: int64

In [18]:


all_dates = pd.date_range(df["ds"].min(), df["ds"].max(), freq = "D")
equipos = sorted(df["equipo"].unique())

full_index = pd.MultiIndex.from_product(
    [all_dates, equipos],
    names = ["ds", "equipo"]
)

df = (df.set_index(["ds", "equipo"])
      .reindex(full_index)
      .reset_index()
     )

In [19]:
print(df.columns)

Index(['ds', 'equipo', 'y', 'capacidad', 'tipo_dia', 'novedad'], dtype='object')


In [20]:


all_dates = pd.date_range(df["ds"].min(), df["ds"].max(), freq = "D")
equipos = sorted(df["equipo"].unique())

full_index = pd.MultiIndex.from_product(
    [all_dates, equipos],
    names = ["ds", "equipo"]
)

df = (df.set_index(["ds", "equipo"])
      .reindex(full_index)
      .reset_index()
     )

In [21]:
df = df.sort_values(["equipo", "ds"])


In [22]:
print(df.head(4))

           ds      equipo   y  capacidad tipo_dia novedad
0  2022-05-03  EDGE1_6107 NaN        NaN      NaN     NaN
9  2022-05-04  EDGE1_6107 NaN        NaN      NaN     NaN
18 2022-05-05  EDGE1_6107 NaN        NaN      NaN     NaN
27 2022-05-06  EDGE1_6107 NaN        NaN      NaN     NaN


In [23]:
df["capacidad"] = (
    df.groupby("equipo")["capacidad"]
    .ffill()
    .bfill()
    .fillna(0)
    .astype(int)
)

df["y"] = (
    df.groupby("equipo")["y"]
    .ffill()
    .bfill()
    .fillna(0)
    .astype(int)
)



In [24]:
for col in ["novedad_Falla", "novedad_Mantenimiento", "novedad_ninguna"]:
    if col in df.columns:
        df[col] = df[col].astype("boolean").fillna(False).astype(bool)

In [25]:
df.groupby('equipo')['ds'].size()

equipo
EDGE1_6107    1459
EDGE2_6137    1459
HALCYON       1459
HALCYON3      1459
HALCYON4      1459
HALCYON5      1459
SYNERGY       1459
SYNERGY2      1459
SYNERGY3      1459
Name: ds, dtype: int64

In [26]:
db_train = pd.read_csv(settings.DATA_DIR / "processed"/ "demanda"/ "train" / "train_demand.csv")

print(db_train.head(3))

           ds      equipo   y  capacidad  festivo  fin_de_semana  \
0  2022-05-03  EDGE1_6107  59         70    False          False   
1  2022-05-03  EDGE2_6137  59         70    False          False   
2  2022-05-03     HALCYON  59         70    False          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  
1               False  False          False  
2               False  False          False  


In [27]:
db_val = pd.read_csv(settings.DATA_DIR / "processed" / "demanda" / "validation"/ "validation_demand.csv")

print(db_val.head(3))

           ds      equipo  y  capacidad  festivo  fin_de_semana  \
0  2026-01-01  EDGE1_6107  0          0     True          False   
1  2026-01-01  EDGE2_6137  0          0     True          False   
2  2026-01-01     HALCYON  0          0     True          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  
1               False  False          False  
2               False  False          False  


In [28]:
db_train_dates = pd.read_csv(settings.DATA_DIR / "processed"/ "demanda" / "train" / "train_dates.csv")
print(db_train_dates.head(5))

           ds
0  2022-05-03
1  2022-05-04
2  2022-05-05
3  2022-05-06
4  2022-05-07


In [29]:
db_val_dates = pd.read_csv(settings.DATA_DIR / "processed"/ "demanda" / "validation" / "val_dates.csv")
print(db_val_dates.head(5))

           ds
0  2026-01-01
1  2026-01-02
2  2026-01-03
3  2026-01-04
4  2026-01-05


In [30]:
#train_dates = db_train_dates["ds"].unique()
val_dates = db_val_dates["ds"].unique()
print(type(db_train_dates))
print(db_train_dates)


<class 'pandas.core.frame.DataFrame'>
              ds
0     2022-05-03
1     2022-05-04
2     2022-05-05
3     2022-05-06
4     2022-05-07
...          ...
1334  2025-12-27
1335  2025-12-28
1336  2025-12-29
1337  2025-12-30
1338  2025-12-31

[1339 rows x 1 columns]


In [31]:
db_full_train = pd.read_csv(settings.DATA_DIR / "processed" / "demanda"/ "produccion_radioterapia.csv")
print(db_full_train.shape)
print(db_full_train.columns)

(13131, 9)
Index(['ds', 'equipo', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


In [32]:
db_full_train.groupby('equipo')['ds'].size()

equipo
EDGE1_6107    1459
EDGE2_6137    1459
HALCYON       1459
HALCYON3      1459
HALCYON4      1459
HALCYON5      1459
SYNERGY       1459
SYNERGY2      1459
SYNERGY3      1459
Name: ds, dtype: int64

In [33]:
equipos = sorted(df['equipo'].unique())
print(equipos)

['EDGE1_6107', 'EDGE2_6137', 'HALCYON', 'HALCYON3', 'HALCYON4', 'HALCYON5', 'SYNERGY', 'SYNERGY2', 'SYNERGY3']


In [34]:
print(df.columns)

Index(['ds', 'equipo', 'y', 'capacidad', 'tipo_dia', 'novedad'], dtype='object')


In [35]:
print(df.shape)
print(df.head(5))

(13131, 6)
           ds      equipo  y  capacidad tipo_dia novedad
0  2022-05-03  EDGE1_6107  0          0      NaN     NaN
9  2022-05-04  EDGE1_6107  0          0      NaN     NaN
18 2022-05-05  EDGE1_6107  0          0      NaN     NaN
27 2022-05-06  EDGE1_6107  0          0      NaN     NaN
36 2022-05-07  EDGE1_6107  0          0      NaN     NaN


# Test funciones del pipeline

In [36]:
path_raw = settings.DATA_DIR / "raw" / "SESIONES_RADIOTERAPIA_5MAY22-30ABR2026.xlsx"
print(path_raw)
path_output = settings.OUTPUT_PATH / "demanda" / "produccion_radioterapia.csv"
print(path_output)

/home/jorge/Projects/Radioterapia/data/raw/SESIONES_RADIOTERAPIA_5MAY22-30ABR2026.xlsx
/home/jorge/Projects/Radioterapia/data/processed/demanda/produccion_radioterapia.csv


In [37]:
crea_produccion.create_df_produccion_x_equipo(path_raw, path_output)

,ds,equipo,y,capacidad,festivo,fin_de_semana,control_de_calidad,falla,mantenimiento
0,2022-05-03,EDGE1_6107,59,70,False,False,False,False,False
1,2022-05-03,EDGE2_6137,59,70,False,False,False,False,False
2,2022-05-03,HALCYON,59,70,False,False,False,False,False
3,2022-05-03,HALCYON3,41,65,False,False,False,False,False
4,2022-05-03,HALCYON4,41,65,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
13126,2026-04-30,HALCYON4,0,0,True,False,False,False,False
13127,2026-04-30,HALCYON5,0,0,True,False,False,False,False
13128,2026-04-30,SYNERGY,0,0,False,False,False,False,False
13129,2026-04-30,SYNERGY2,13,47,False,False,False,False,False


In [38]:
produccion_path = settings.OUTPUT_PATH / "demanda" / "produccion_radioterapia.csv"
print(produccion_path)
df_produccion = pd.read_csv(produccion_path)

/home/jorge/Projects/Radioterapia/data/processed/demanda/produccion_radioterapia.csv


In [39]:
df_produccion1 = df_produccion.copy()

In [40]:
print(df_produccion1.shape)

(13131, 9)


In [41]:
print(df_produccion1.head(4))

           ds      equipo   y  capacidad  festivo  fin_de_semana  \
0  2022-05-03  EDGE1_6107  59         70    False          False   
1  2022-05-03  EDGE2_6137  59         70    False          False   
2  2022-05-03     HALCYON  59         70    False          False   
3  2022-05-03    HALCYON3  41         65    False          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  
1               False  False          False  
2               False  False          False  
3               False  False          False  


In [42]:
demanda_diaria1 = df_produccion1.groupby("ds", as_index = False).agg(
    y = ("y", "sum"),
    capacidad = ("capacidad", "sum"),
    festivo = ("festivo", lambda x: x.mode()[0]),
    fin_de_semana = ("fin_de_semana", lambda x: x.mode()[0]),
    #laboral = ("laboral", lambda x: x.mode()[0]),
    control_de_calidad = ("control_de_calidad", lambda x: x.mode()[0]),
    falla = ("falla", lambda x: x.mode()[0]),
    mantenimiento = ("mantenimiento", lambda x: x.mode()[0]),
    #ninguna = ("ninguna", lambda x: x.mode()[0]),
)
print(demanda_diaria1.head(4))

           ds    y  capacidad  festivo  fin_de_semana  control_de_calidad  \
0  2022-05-03  407        564    False          False               False   
1  2022-05-04  443        564    False          False               False   
2  2022-05-05  451        564    False          False               False   
3  2022-05-06  379        564    False          False               False   

   falla  mantenimiento  
0  False          False  
1  False          False  
2  False          False  
3  False          False  


In [43]:
df_produccion = extract.read_produccion(produccion_path)
print(df_produccion)

              ds      equipo   y  capacidad  festivo  fin_de_semana  \
0     2022-05-03  EDGE1_6107  59         70    False          False   
1     2022-05-03  EDGE2_6137  59         70    False          False   
2     2022-05-03     HALCYON  59         70    False          False   
3     2022-05-03    HALCYON3  41         65    False          False   
4     2022-05-03    HALCYON4  41         65    False          False   
...          ...         ...  ..        ...      ...            ...   
13126 2026-04-30    HALCYON4   0          0     True          False   
13127 2026-04-30    HALCYON5   0          0     True          False   
13128 2026-04-30     SYNERGY   0          0    False          False   
13129 2026-04-30    SYNERGY2  13         47    False          False   
13130 2026-04-30    SYNERGY3   0          0    False          False   

       control_de_calidad  falla  mantenimiento  
0                   False  False          False  
1                   False  False          False

In [44]:
path_demanda_diaria = extract.get_demanda_diaria(df_produccion)

In [45]:
print(path_demanda_diaria)

/home/jorge/Projects/Radioterapia/data/processed/demanda/demanda_diaria_total.csv


In [46]:
path_demanda_hist = extract.get_demanda_hist_equipment(df_produccion)

In [47]:
print(path_demanda_hist)

/home/jorge/Projects/Radioterapia/data/processed/demanda/demanda_diaria_x_equipo.csv


In [48]:
print(settings.days_to_forecast)

30


In [49]:
os.makedirs(os.path.dirname(os.path.join(settings.OUTPUT_PATH, "demanda", "train")),exist_ok = True)
os.makedirs(os.path.dirname(os.path.join(settings.OUTPUT_PATH, "demanda", "test")), exist_ok = True)
os.makedirs(os.path.dirname(os.path.join(settings.OUTPUT_PATH, "demanda", "validation")), exist_ok = True)
path_train = os.path.join(settings.OUTPUT_PATH, "demanda", "train")
path_test = os.path.join(settings.OUTPUT_PATH, "demanda", "test")
path_val = os.path.join(settings.OUTPUT_PATH, "demanda", "validation")


path_demanda_hist = extract.get_demanda_hist_equipment(df_produccion)
df_hist = pd.read_csv(path_demanda_hist)
df_total = pd.read_csv(path_demanda_diaria)


In [50]:
print(df_hist.head(5))

           ds      equipo   y  capacidad  festivo  fin_de_semana  \
0  2022-05-03  EDGE1_6107  59         70    False          False   
1  2022-05-03  EDGE2_6137  59         70    False          False   
2  2022-05-03     HALCYON  59         70    False          False   
3  2022-05-03    HALCYON3  41         65    False          False   
4  2022-05-03    HALCYON4  41         65    False          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  
1               False  False          False  
2               False  False          False  
3               False  False          False  
4               False  False          False  


In [51]:
train,val,test,train_dates,val_dates,test_dates = filtra.train_val_test(df_hist, settings.days_to_forecast*2)

In [52]:
print(train.head(1))

           ds      equipo   y  capacidad  festivo  fin_de_semana  \
0  2022-05-03  EDGE1_6107  59         70    False          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  


In [53]:
print(val.head(1))

               ds      equipo  y  capacidad  festivo  fin_de_semana  \
12051  2026-01-01  EDGE1_6107  0          0     True          False   

       control_de_calidad  falla  mantenimiento  
12051               False  False          False  


In [54]:
print(test.head(1))

               ds      equipo   y  capacidad  festivo  fin_de_semana  \
12591  2026-03-02  EDGE1_6107  18         21    False          False   

       control_de_calidad  falla  mantenimiento  
12591               False  False          False  


In [55]:
print(path_train)

/home/jorge/Projects/Radioterapia/data/processed/demanda/train


In [56]:



train.to_csv(os.path.join(path_train,"train_demand_csv"), index = False)
np.savetxt(os.path.join(path_train,"train_dates.csv"), train_dates.astype(str), 
           delimiter = ",",
          fmt = "%s")


test.to_csv(os.path.join(path_test,"test_demand.csv"), index = False)
pd.Series(test_dates, name="ds").to_csv(os.path.join(path_val,"test_dates.csv"), 
                                                    index = False)

val.to_csv(os.path.join(path_val, "validation_demand.csv"), index = False)
pd.Series(val_dates, name="ds").to_csv(os.path.join(path_val,"val_dates.csv"), 
                                                    index = False)

In [57]:
print(type(train_dates))

<class 'numpy.ndarray'>


In [58]:
train, val, test, train_dates, val_dates, test_dates = filtra.train_val_test(df_total,60)

In [59]:
print(len(train))
print(len(val))
print(len(test))
print(type(train_dates))

1339
60
60
<class 'numpy.ndarray'>


In [60]:
print(train)

              ds    y  capacidad  festivo  fin_de_semana  control_de_calidad  \
0     2022-05-03  407        564    False          False               False   
1     2022-05-04  443        564    False          False               False   
2     2022-05-05  451        564    False          False               False   
3     2022-05-06  379        564    False          False               False   
4     2022-05-07    0          0    False          False               False   
...          ...  ...        ...      ...            ...                 ...   
1334  2025-12-27    0          0    False           True               False   
1335  2025-12-28    0          0    False           True               False   
1336  2025-12-29  173        352    False          False               False   
1337  2025-12-30  166        352    False          False               False   
1338  2025-12-31  142        352    False          False               False   

      falla  mantenimiento  
0     Fals

## Fit final models

In [61]:
param_grid = {
    "changepoint_prior_scale": [0.01, 0.1, 0.5],
    "seasonality_prior_scale": [1, 5, 10] }


In [62]:
regressors = [c for c in train.columns if c not in ['ds','y','equipo']]

In [63]:
print(regressors)

['capacidad', 'festivo', 'fin_de_semana', 'control_de_calidad', 'falla', 'mantenimiento']


In [64]:
train = pd.read_csv(settings.DATA_DIR / "processed"/ "demanda" / "train"/ "train_demand.csv")

In [65]:
print(train.columns)

Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


In [66]:
best_params = fcProphet.tune_total(train, val, regressors, param_grid)

06:25:06 - cmdstanpy - INFO - Chain [1] start processing
06:25:07 - cmdstanpy - INFO - Chain [1] done processing
06:25:07 - cmdstanpy - INFO - Chain [1] start processing
06:25:07 - cmdstanpy - INFO - Chain [1] done processing
06:25:07 - cmdstanpy - INFO - Chain [1] start processing
06:25:08 - cmdstanpy - INFO - Chain [1] done processing
06:25:08 - cmdstanpy - INFO - Chain [1] start processing
06:25:08 - cmdstanpy - INFO - Chain [1] done processing
06:25:09 - cmdstanpy - INFO - Chain [1] start processing
06:25:09 - cmdstanpy - INFO - Chain [1] done processing
06:25:10 - cmdstanpy - INFO - Chain [1] start processing
06:25:10 - cmdstanpy - INFO - Chain [1] done processing
06:25:10 - cmdstanpy - INFO - Chain [1] start processing
06:25:11 - cmdstanpy - INFO - Chain [1] done processing
06:25:11 - cmdstanpy - INFO - Chain [1] start processing
06:25:12 - cmdstanpy - INFO - Chain [1] done processing
06:25:12 - cmdstanpy - INFO - Chain [1] start processing
06:25:13 - cmdstanpy - INFO - Chain [1]

In [67]:
train_full = df_hist[df_hist["ds"].isin(np.concatenate([train_dates, val_dates]))]
test = df_hist[df_hist["ds"].isin(test_dates)]

In [68]:
print(train_full.columns)

Index(['ds', 'equipo', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


In [69]:
models, residuals, results = fcProphet.bottom_up_models(train_full, best_params, regressors, equipos)

Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


06:25:18 - cmdstanpy - INFO - Chain [1] start processing
06:25:18 - cmdstanpy - INFO - Chain [1] done processing
06:25:19 - cmdstanpy - INFO - Chain [1] start processing


Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


06:25:19 - cmdstanpy - INFO - Chain [1] done processing
06:25:19 - cmdstanpy - INFO - Chain [1] start processing


Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


...

06:25:22 - cmdstanpy - INFO - Chain [1] done processing
06:25:23 - cmdstanpy - INFO - Chain [1] start processing


Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


06:25:23 - cmdstanpy - INFO - Chain [1] done processing
06:25:24 - cmdstanpy - INFO - Chain [1] start processing


Index(['ds', 'y', 'capacidad', 'festivo', 'fin_de_semana',
       'control_de_calidad', 'falla', 'mantenimiento'],
      dtype='object')


06:25:24 - cmdstanpy - INFO - Chain [1] done processing


In [70]:
print(models)

{'EDGE1_6107': <prophet.forecaster.Prophet object at 0x7d85833cb4c0>, 'EDGE2_6137': <prophet.forecaster.Prophet object at 0x7d8582fb3970>, 'HALCYON': <prophet.forecaster.Prophet object at 0x7d8582fb1270>, 'HALCYON3': <prophet.forecaster.Prophet object at 0x7d8582fdeef0>, 'HALCYON4': <prophet.forecaster.Prophet object at 0x7d8582fb6110>, 'HALCYON5': <prophet.forecaster.Prophet object at 0x7d8582f73460>, 'SYNERGY': <prophet.forecaster.Prophet object at 0x7d8582f72650>, 'SYNERGY2': <prophet.forecaster.Prophet object at 0x7d8582fb4190>, 'SYNERGY3': <prophet.forecaster.Prophet object at 0x7d8582d11b40>}


In [71]:
print(residuals)
print(len(residuals))

[array([-1.08374661,  3.01623304,  2.70280319, ..., -5.37524474,
       -1.6604827 , -1.82877635], shape=(1399,)), array([-2.31092667,  2.29476154,  2.14146716, ...,  2.56435759,
       -5.9463928 , -6.20150924], shape=(1399,)), array([0.36142298, 4.49140245, 4.46078705, ..., 6.79808219, 5.1296372 ,
       4.93106673], shape=(1399,)), array([-10.16120538,  -5.67451237,  -4.73319939, ...,   9.06600545,
         3.11187822,   2.93166522], shape=(1399,)), array([-7.76310574, -3.92248678, -2.83259289, ...,  8.05391106,
       -1.74031642, -1.69208505], shape=(1399,)), array([ -9.40261403,  -4.56328828,  -3.42571099, ..., -16.93092113,
         1.1162593 ,   1.04569826], shape=(1399,)), array([-9.93433373, -5.33945487, -3.89370249, ...,  2.93513953,
        0.62731087,  0.56844096], shape=(1399,)), array([-4.69983008,  0.11792438,  2.4378011 , ..., -1.62856442,
        0.74641916,  0.7161795 ], shape=(1399,)), array([ -0.31127053,  -0.85331951,   3.84884038, ..., -13.64700725,
         4.23

In [72]:
print(results["residual"])
print(results["residual"].shape)

0        -1.083747
1         3.016233
2         2.702803
3        -4.883727
4        -5.772648
           ...    
12586    -5.093952
12587    -3.347794
12588   -13.647007
12589     0.000000
12590     0.000000
Name: residual, Length: 12591, dtype: float64
(12591,)


In [74]:
residual_m = np.array(residuals)
residual_flat = residual_m.flatten()

compare_residuals = pd.DataFrame({"residuals": residual_flat,
"results_residuals": results["residual"].values
})
compare_residuals["residuals_diff"] = compare_residuals["residuals"]-compare_residuals["results_residuals"]
print(compare_residuals["residuals_diff"].sum())
print(compare_residuals)

8003.409987072195
       residuals  results_residuals  residuals_diff
0      -1.083747          -1.083747        0.000000
1       3.016233           3.016233        0.000000
2       2.702803           2.702803        0.000000
3      -4.883727          -4.883727        0.000000
4      -5.772648          -5.772648        0.000000
...          ...                ...             ...
12586  -5.093952          -5.093952        0.000000
12587  -3.347794          -3.347794        0.000000
12588 -13.647007         -13.647007        0.000000
12589   4.236750           0.000000        4.236750
12590   4.124078           0.000000        4.124078

[12591 rows x 3 columns]


In [75]:
print(results)

               ds      equipo  festivo  fin_de_semana  control_de_calidad  \
0      2022-05-03  EDGE1_6107    False          False               False   
1      2022-05-04  EDGE1_6107    False          False               False   
2      2022-05-05  EDGE1_6107    False          False               False   
3      2022-05-06  EDGE1_6107    False          False               False   
4      2022-05-07  EDGE1_6107    False          False               False   
...           ...         ...      ...            ...                 ...   
12586  2026-02-25    SYNERGY3    False          False               False   
12587  2026-02-26    SYNERGY3    False          False               False   
12588  2026-02-27    SYNERGY3    False          False               False   
12589  2026-02-28    SYNERGY3    False           True               False   
12590  2026-03-01    SYNERGY3    False           True               False   

       falla  mantenimiento  capacidad   y       yhat   residual  
0      F

## Covariance Matrices and Standard Mint

In [76]:

pred_total_train, train_total_fc_and_pred, res_total, m_total = fcProphet.top_down_total(train_full,best_params ,regressors)


06:25:46 - cmdstanpy - INFO - Chain [1] start processing
06:25:47 - cmdstanpy - INFO - Chain [1] done processing


In [77]:
print(m_total)

In [78]:
print(len(res_total))
print(residuals)

1399
[array([-1.08374661,  3.01623304,  2.70280319, ..., -5.37524474,
       -1.6604827 , -1.82877635], shape=(1399,)), array([-2.31092667,  2.29476154,  2.14146716, ...,  2.56435759,
       -5.9463928 , -6.20150924], shape=(1399,)), array([0.36142298, 4.49140245, 4.46078705, ..., 6.79808219, 5.1296372 ,
       4.93106673], shape=(1399,)), array([-10.16120538,  -5.67451237,  -4.73319939, ...,   9.06600545,
         3.11187822,   2.93166522], shape=(1399,)), array([-7.76310574, -3.92248678, -2.83259289, ...,  8.05391106,
       -1.74031642, -1.69208505], shape=(1399,)), array([ -9.40261403,  -4.56328828,  -3.42571099, ..., -16.93092113,
         1.1162593 ,   1.04569826], shape=(1399,)), array([-9.93433373, -5.33945487, -3.89370249, ...,  2.93513953,
        0.62731087,  0.56844096], shape=(1399,)), array([-4.69983008,  0.11792438,  2.4378011 , ..., -1.62856442,
        0.74641916,  0.7161795 ], shape=(1399,)), array([ -0.31127053,  -0.85331951,   3.84884038, ..., -13.64700725,
        

In [79]:
residual_matrix = np.vstack([res_total] + residuals)
print(residual_matrix.shape)

(10, 1399)


In [80]:
print(pred_total_train)
print(pred_total_train.shape)


             ds       trend  yhat_lower  yhat_upper  trend_lower  trend_upper  \
0    2022-05-03  289.333875  408.965811  496.310304   289.333875   289.333875   
1    2022-05-04  289.309516  412.791528  495.383436   289.309516   289.309516   
2    2022-05-05  289.285157  406.877959  493.035453   289.285157   289.285157   
3    2022-05-06  289.260798  396.935070  481.751544   289.260798   289.260798   
4    2022-05-07  289.236439  -27.285973   59.440678   289.236439   289.236439   
...         ...         ...         ...         ...          ...          ...   
1394 2026-02-25  193.401114  162.857261  246.073286   193.401114   193.401114   
1395 2026-02-26  193.544662  114.444870  198.229037   193.544662   193.544662   
1396 2026-02-27  193.688211  150.684847  236.063359   193.688211   193.688211   
1397 2026-02-28  193.831760   -8.606460   72.871864   193.831760   193.831760   
1398 2026-03-01  193.975309   -9.593913   76.180257   193.975309   193.975309   

      additive_terms  addit

In [81]:
print(train_total_fc_and_pred)
print(train_total_fc_and_pred.shape)
print(type(train_total_fc_and_pred))

             ds    y        yhat  residuals
0    2022-05-03  407  452.971519 -45.971519
1    2022-05-04  443  454.581204 -11.581204
2    2022-05-05  451  450.120904   0.879096
3    2022-05-06  379  438.222175 -59.222175
4    2022-05-07    0   11.839444 -11.839444
...         ...  ...         ...        ...
1394 2026-02-25  209  204.060891   4.939109
1395 2026-02-26  205  156.621674  48.378326
1396 2026-02-27  186  191.854469  -5.854469
1397 2026-02-28   13   32.281469 -19.281469
1398 2026-03-01   13   33.403239 -20.403239

[1399 rows x 4 columns]
(1399, 4)
<class 'pandas.core.frame.DataFrame'>


In [82]:
print(pred_total_train.head(4))
print(pred_total_train.shape)
print(type(pred_total_train))

          ds       trend  yhat_lower  yhat_upper  trend_lower  trend_upper  \
0 2022-05-03  289.333875  408.965811  496.310304   289.333875   289.333875   
1 2022-05-04  289.309516  412.791528  495.383436   289.309516   289.309516   
2 2022-05-05  289.285157  406.877959  493.035453   289.285157   289.285157   
3 2022-05-06  289.260798  396.935070  481.751544   289.260798   289.260798   

   additive_terms  additive_terms_lower  additive_terms_upper   capacidad  \
0      163.637644            163.637644            163.637644  180.816094   
1      165.271688            165.271688            165.271688  180.816094   
2      160.835747            160.835747            160.835747  180.816094   
3      148.961377            148.961377            148.961377  180.816094   

   ...     weekly  weekly_lower  weekly_upper    yearly  yearly_lower  \
0  ...  34.098588     34.098588     34.098588  7.317521      7.317521   
1  ...  36.965064     36.965064     36.965064  6.085089      6.085089   
2  .

In [83]:
print(res_total)
print(res_total.shape)
print(type(res_total))

[-45.97151888 -11.58120428   0.87909577 ...  -5.85446854 -19.28146869
 -20.40323909]
(1399,)
<class 'numpy.ndarray'>


In [84]:
print(m_total)
print(type(m_total))

<class 'prophet.forecaster.Prophet'>


In [85]:
results

,ds,equipo,festivo,fin_de_semana,control_de_calidad,falla,mantenimiento,capacidad,y,yhat,residual
0,2022-05-03,EDGE1_6107,False,False,False,False,False,70,59,60.083747,-1.083747
1,2022-05-04,EDGE1_6107,False,False,False,False,False,70,64,60.983767,3.016233
2,2022-05-05,EDGE1_6107,False,False,False,False,False,70,63,60.297197,2.702803
3,2022-05-06,EDGE1_6107,False,False,False,False,False,70,55,59.883727,-4.883727
4,2022-05-07,EDGE1_6107,False,False,False,False,False,0,0,5.772648,-5.772648
...,...,...,...,...,...,...,...,...,...,...,...
12586,2026-02-25,SYNERGY3,False,False,False,False,False,30,9,14.093952,-5.093952
12587,2026-02-26,SYNERGY3,False,False,False,False,False,30,11,14.347794,-3.347794
12588,2026-02-27,SYNERGY3,False,False,False,False,False,30,1,14.647007,-13.647007
12589,2026-02-28,SYNERGY3,False,True,False,False,False,0,0,0.000000,0.000000


In [86]:
results.pivot(index = "ds", columns = "equipo", values = "residual")

equipo,EDGE1_6107,EDGE2_6137,HALCYON,HALCYON3,HALCYON4,HALCYON5,SYNERGY,SYNERGY2,SYNERGY3
ds,,,,,,,,,
2022-05-03,-1.083747,-2.310927,0.361423,-10.161205,-7.763106,-9.402614,-9.934334,-4.699830,-0.311271
2022-05-04,3.016233,2.294762,4.491402,-5.674512,-3.922487,-4.563288,-5.339455,0.117924,-0.853320
2022-05-05,2.702803,2.141467,4.460787,-4.733199,-2.832593,-3.425711,-3.893702,2.437801,3.848840
2022-05-06,-4.883727,-5.992408,-1.929453,-5.814438,-3.810004,-3.865575,-5.119132,1.609211,-30.500409
2022-05-07,-5.772648,-9.044375,-9.876124,-1.817219,-0.376726,-1.921141,-5.002133,-9.777074,-11.068120
...,...,...,...,...,...,...,...,...,...
2026-02-25,-1.784467,4.700322,11.128180,6.060513,13.932280,-22.375436,0.000000,-2.847330,-5.093952
2026-02-26,-4.452796,3.113827,16.625429,16.572513,24.088383,0.000000,0.000000,-2.667047,-3.347794
2026-02-27,-5.375245,2.564358,6.798082,9.066005,8.053911,-16.930921,0.000000,-1.628564,-13.647007


In [87]:
print(equipos)
residuals_matrix,residuals_df = fcProphet.cal_residual_matrix(res_total, residuals, equipos)

['EDGE1_6107', 'EDGE2_6137', 'HALCYON', 'HALCYON3', 'HALCYON4', 'HALCYON5', 'SYNERGY', 'SYNERGY2', 'SYNERGY3']


In [88]:
print(residuals_df.shape)

(10, 1399)


In [89]:
print(residuals_df)
print(type(residuals_df))
np.cov(residuals_df).shape


                 0          1         2          3          4          5     \
total      -45.971519 -11.581204  0.879096 -59.222175 -11.839444 -11.329637   
EDGE1_6107  -1.083747   3.016233  2.702803  -4.883727  -5.772648  -5.655578   
EDGE2_6137  -2.310927   2.294762  2.141467  -5.992408  -9.044375  -8.936298   
HALCYON      0.361423   4.491402  4.460787  -1.929453  -9.876124  -9.726857   
HALCYON3   -10.161205  -5.674512 -4.733199  -5.814438  -1.817219  -1.598829   
HALCYON4    -7.763106  -3.922487 -2.832593  -3.810004  -0.376726  -0.435253   
HALCYON5    -9.402614  -4.563288 -3.425711  -3.865575  -1.921141  -1.875803   
SYNERGY     -9.934334  -5.339455 -3.893702  -5.119132  -5.002133  -4.848819   
SYNERGY2    -4.699830   0.117924  2.437801   1.609211  -9.777074  -9.689878   
SYNERGY3    -0.311271  -0.853320  3.848840 -30.500409 -11.068120 -11.238535   

                6         7         8          9     ...       1389  \
total      -6.627527  7.300485  8.488658 -33.260894  ... -1

(10, 10)

In [90]:
W, lw, W_shrink = fcProphet.compute_covariance_matrices(residuals_df)

In [91]:
S = fcProphet.summing_matrix(equipos) 

In [92]:
from numpy.linalg import inv
print(S.T.shape) 
print(S.T)
print(W.shape)

(9, 10)
[[1. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
(10, 10)


In [93]:
print(W.shape)
print(S.shape)
print(test.head(5))

(10, 10)
(10, 9)
               ds      equipo   y  capacidad  festivo  fin_de_semana  \
12591  2026-03-02  EDGE1_6107  18         21    False          False   
12592  2026-03-02  EDGE2_6137  13         36    False          False   
12593  2026-03-02     HALCYON  36         55    False          False   
12594  2026-03-02    HALCYON3  26         65    False          False   
12595  2026-03-02    HALCYON4  42         65    False          False   

       control_de_calidad  falla  mantenimiento  
12591               False  False          False  
12592               False  False          False  
12593               False  False          False  
12594               False  False          False  
12595               False  False          False  


In [94]:
test[test["equipo"] == equipos[0]] [["ds"]+regressors]

,ds,capacidad,festivo,fin_de_semana,control_de_calidad,falla,mantenimiento
12591,2026-03-02,21,False,False,False,False,False
12600,2026-03-03,21,False,False,False,False,False
12609,2026-03-04,21,False,False,False,False,False
12618,2026-03-05,21,False,False,False,False,False
12627,2026-03-06,21,False,False,False,False,False
12636,2026-03-07,0,False,True,False,False,False
12645,2026-03-08,0,False,True,False,False,False
12654,2026-03-09,21,False,False,False,False,False
12663,2026-03-10,21,False,False,False,False,False
12672,2026-03-11,21,False,False,False,False,False


In [95]:
test[test["equipo"] == equipos[0]]["y"].values

array([18, 11, 17, 12, 11,  0,  0, 12, 14, 13, 12,  9,  0,  0, 22, 16, 15,
       20, 14,  0,  0,  5, 14, 11, 12, 12,  0,  0, 26, 13, 13, 13,  0,  0,
        0, 14, 16, 13,  9,  8,  0,  0, 11,  8, 23, 10, 13,  0,  0,  8,  6,
        7, 10, 20,  0,  0, 20, 21, 20,  0])

In [96]:
results_test_df, Yhat, total_pred = fcProphet.fc_bottom_up_test(test_dates, test, m_total, models, equipos, regressors)

In [97]:
print(results_test_df.head(5))

           ds      equipo  festivo  fin_de_semana  control_de_calidad  falla  \
0  2026-03-02  EDGE1_6107    False          False               False  False   
1  2026-03-03  EDGE1_6107    False          False               False  False   
2  2026-03-04  EDGE1_6107    False          False               False  False   
3  2026-03-05  EDGE1_6107    False          False               False  False   
4  2026-03-06  EDGE1_6107    False          False               False  False   

   mantenimiento  capacidad   y       yhat  residual  
0          False         21  18  13.513211  4.486789  
1          False         21  11  13.729597 -2.729597  
2          False         21  17  14.916105  2.083895  
3          False         21  12  14.485577 -2.485577  
4          False         21  11  14.294469 -3.294469  


In [98]:
print(Yhat)

[[ 1.91704542e+02  1.97973952e+02  2.06669493e+02  2.04156578e+02
   1.93960714e+02  3.85223712e+01  3.91874391e+01  2.02943130e+02
   2.07601065e+02  2.10799794e+02  2.07700937e+02  1.97988402e+02
   4.09601001e+01  4.11435818e+01  2.09405618e+02  2.08827652e+02
   2.11809147e+02  2.08607795e+02  1.98916074e+02  4.20370973e+01
   4.25000178e+01  2.06253926e+02  2.11121004e+02  2.14744467e+02
   2.12284417e+02  1.85853614e+02  1.77975406e+01  1.91902617e+01
   1.77486838e+02  1.97904212e+02  2.02436600e+02  2.00827340e+02
   4.14079188e+01  2.91933033e+01  3.11176899e+01  2.08268127e+02
   2.15349748e+02  2.15008587e+02  2.13255508e+02  2.08682856e+02
   4.07627154e+01  4.19763879e+01  2.13323216e+02  2.19352446e+02
   2.27642664e+02  2.20701744e+02  2.34361910e+02  4.42194814e+01
   4.38789183e+01  2.13652876e+02  2.18120834e+02  1.81653129e+02
   1.77125052e+02  2.04234572e+02  3.74443063e+01  3.60252011e+01
   2.04887093e+02  2.08625144e+02  2.09944090e+02  5.48154601e+01]
 [ 1.3513

In [99]:
Yhat < 0

array([[False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False],
       [False, False, False, False, False, Fal

In [100]:
from numpy.linalg import inv
inv(W)

array([[ 5.14312714e-03, -8.00510802e-03, -6.93128622e-03,
        -9.24212252e-04, -4.00128356e-03, -4.86892310e-03,
        -4.95838006e-03, -5.23564905e-03, -8.92471930e-04,
        -3.12618623e-04],
       [-8.00510802e-03,  8.72279652e-02, -9.91608885e-03,
        -1.97791077e-02,  1.62778258e-02,  1.64664185e-02,
         4.55478125e-03, -3.23305229e-03,  3.42161023e-03,
        -5.62396045e-04],
       [-6.93128622e-03, -9.91608885e-03,  4.56705791e-02,
        -9.28425715e-03,  1.99184055e-03, -9.96468867e-05,
         7.94654112e-03,  1.22496548e-02,  4.92912817e-04,
        -6.43377746e-03],
       [-9.24212252e-04, -1.97791077e-02, -9.28425715e-03,
         4.75340900e-02, -4.93359490e-03, -2.89388602e-03,
         3.89669431e-03,  7.26076397e-03, -7.51920268e-03,
        -9.47408118e-03],
       [-4.00128356e-03,  1.62778258e-02,  1.99184055e-03,
        -4.93359490e-03,  5.23076207e-02, -2.46754168e-02,
        -4.75764054e-03,  7.33242246e-04, -1.21724811e-03,
         1.

In [101]:
YminT = fcProphet.mint_reconcile(Yhat,W,S)

In [210]:
print(YminT)
print(YminT.shape)

[[ 1.72872798e+02  1.77174696e+02  1.80474662e+02  1.78354454e+02
   1.73557744e+02  3.48999729e+00  3.50065319e+00  1.69241127e+02
   1.78669093e+02  1.80629893e+02  1.78146247e+02  1.67697504e+02
   2.64799598e+00  2.41311798e+00  1.68867348e+02  1.77285348e+02
   1.79211008e+02  1.76774335e+02  1.66416471e+02  1.62252932e+00
   1.68397975e+00  1.69321952e+02  1.77368738e+02  1.79793143e+02
   1.77901858e+02  1.63004213e+02 -1.71104313e+01 -1.64486397e+01
   1.45597807e+02  1.65355338e+02  1.68293301e+02  1.66855925e+02
   6.37699532e+00 -1.14176525e+01 -1.05561217e+01  1.65754377e+02
   1.70553984e+02  1.74145791e+02  1.72472801e+02  1.67307491e+02
  -6.59342791e+00 -6.27557183e+00  1.70183863e+02  1.74281852e+02
   1.77391115e+02  1.68504463e+02  1.67177633e+02 -6.87134638e+00
  -7.38092756e+00  1.68277847e+02  1.71618688e+02  1.44701494e+02
   1.41601127e+02  1.60554580e+02 -1.18396476e+01 -1.26889415e+01
   1.62739021e+02  1.65960613e+02  1.67308875e+02  7.87376872e+00]
 [ 1.0911

In [215]:
labels = ["total"] + list(equipos)
YminT_df = pd.DataFrame(YminT, index = labels)

In [216]:
print(YminT_df)

                    0           1           2           3           4   \
total       172.872798  177.174696  180.474662  178.354454  173.557744   
EDGE1_6107   10.911264   10.074697   10.930095   10.666319   11.474966   
EDGE2_6137   12.737822   12.262006   12.263013   11.547973   12.398565   
HALCYON      30.546178   32.402617   34.671748   33.834612   31.965866   
HALCYON3     34.911969   34.655774   35.054255   35.782176   36.414757   
HALCYON4     36.650226   36.511843   37.056686   36.710608   36.227328   
HALCYON5     33.724966   34.809372   34.197193   34.277103   33.625245   
SYNERGY      12.053608   12.759273   12.905360   12.590799   10.256139   
SYNERGY2     -6.117475   -5.006070   -3.471604   -3.860584   -5.353369   
SYNERGY3      7.454241    8.705183    6.867916    6.805449    6.548246   

                   5          6           7           8           9   ...  \
total        3.489997   3.500653  169.241127  178.669093  180.629893  ...   
EDGE1_6107  -1.294008  -1.32288

In [217]:
Y_mint_shrk = fcProphet.mint_reconcile(Yhat,W_shrink,S)

In [218]:
print(Y_mint_shrk)
print(Y_mint_shrk.shape)

[[ 1.72861099e+02  1.77164601e+02  1.80467883e+02  1.78347341e+02
   1.73549208e+02  3.47880912e+00  3.48924670e+00  1.69239093e+02
   1.78660956e+02  1.80621658e+02  1.78137601e+02  1.67694277e+02
   2.63513267e+00  2.39997025e+00  1.68867092e+02  1.77275347e+02
   1.79200851e+02  1.76763709e+02  1.66411364e+02  1.60757850e+00
   1.66869169e+00  1.69309436e+02  1.77356493e+02  1.79780692e+02
   1.77888890e+02  1.62989056e+02 -1.71390653e+01 -1.64776513e+01
   1.45590541e+02  1.65340927e+02  1.68278662e+02  1.66840756e+02
   6.36415315e+00 -1.14488565e+01 -1.05876821e+01  1.65746013e+02
   1.70547050e+02  1.74128756e+02  1.72455320e+02  1.67298449e+02
  -6.62672540e+00 -6.30909087e+00  1.70163753e+02  1.74263337e+02
   1.77375601e+02  1.68492780e+02  1.67184830e+02 -6.90549909e+00
  -7.41508591e+00  1.68257315e+02  1.71599971e+02  1.44691640e+02
   1.41591227e+02  1.60535225e+02 -1.18731904e+01 -1.27223163e+01
   1.62719430e+02  1.65942975e+02  1.67291588e+02  7.86046013e+00]
 [ 1.0892

In [219]:
y_mint_shrk_df = pd.DataFrame(Y_mint_shrk, index = labels)
print(y_mint_shrk_df.head(5))

                    0           1           2           3           4   \
total       172.861099  177.164601  180.467883  178.347341  173.549208   
EDGE1_6107   10.892674   10.058656   10.919323   10.655015   11.461403   
EDGE2_6137   12.720126   12.246736   12.252759   11.537213   12.385653   
HALCYON      30.564352   32.418299   34.682279   33.845663   31.979126   
HALCYON3     34.897636   34.643406   35.045950   35.773460   36.404299   

                  5         6           7           8           9   ...  \
total       3.478809  3.489247  169.239093  178.660956  180.621658  ...   
EDGE1_6107 -1.311787 -1.341009   10.778008    9.908929   10.754062  ...   
EDGE2_6137  1.779076  1.758132   12.780809   12.213739   12.178258  ...   
HALCYON    -5.896292 -6.045087   33.670468   32.768189   33.640217  ...   
HALCYON3    0.903056  1.173060   21.424983   36.695568   36.856416  ...   

                    50          51          52          53         54  \
total       171.599971  144.691

In [ ]:
def summarize_group(g):
    g = g.sort_values('ds')
    mask = g['y'].isna()

    return pd.Series({
        'size': len(g),
        'min_date':g['ds'].min(),
        'max_date':g['ds'].max(),
        'first_nan_date': g.loc[mask, 'ds'].iloc[0] if mask.any() else pd.NaT,
        'first_nan_position': np.where(mask)[0][0] if mask.any() else None
    })
summary_pred = results.groupby('equipo').apply(summarize_group)
summary_pred

In [ ]:
summary = df.groupby('equipo').agg(
    size = ('equipo', 'size'),
    min_date = ('ds', 'min'),
    max_date = ('ds', 'max')
)

summary


In [ ]:
W = np.cov(residual_df.T)
W += np.eye(W.shape[0]) * 1e-6

In [ ]:
lw = LedoitWolf()
W_shrink = lw.fit(residual_df.T).covariance_

In [ ]:
n = len(equipos)

S = np.vstack([
    np.ones((1,n)),
    np.eye(n)
])

In [304]:
residuals_df.shape

(10, 1399)

In [301]:
W.shape

(1399, 1399)

In [302]:
S.shape

(10, 9)

In [305]:
inv(W).shape

(1399, 1399)